# Module 2 Solutions: Advanced Linear Algebra

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

In [ ]:
# Ex 1: Eigenvalues/vectors
A = np.array([[2, 1], [1, 2]])
vals, vecs = np.linalg.eig(A)
for i in range(2):
    print(f"λ={vals[i]:.4f}: Av={A @ vecs[:,i]}, λv={vals[i]*vecs[:,i]}")
    print(f"Match? {np.allclose(A @ vecs[:,i], vals[i]*vecs[:,i])}")

In [ ]:
# Ex 2: Eigendecomposition
Q = vecs
Lambda = np.diag(vals)
A_recon = Q @ Lambda @ np.linalg.inv(Q)
print(f"A reconstructed:\n{np.round(A_recon, 4)}")
print(f"Match? {np.allclose(A, A_recon)}")

In [ ]:
# Ex 3: SVD
M = np.array([[3,2,2],[2,3,-2]])
U, s, Vt = np.linalg.svd(M)
S = np.zeros_like(M, dtype=float)
np.fill_diagonal(S, s)
print(f"U @ Σ @ V^T == M? {np.allclose(U @ S @ Vt, M)}")

In [ ]:
# Ex 4: Positive definite check
A = np.array([[5,2],[2,3]])
B = np.array([[1,4],[4,1]])
print(f"A eigenvalues: {np.linalg.eigvalsh(A)} → PD: {np.all(np.linalg.eigvalsh(A) > 0)}")
print(f"B eigenvalues: {np.linalg.eigvalsh(B)} → PD: {np.all(np.linalg.eigvalsh(B) > 0)}")

In [ ]:
# Ex 5: QR decomposition
A = np.array([[1,1,0],[1,0,1],[0,1,1]])
Q, R = np.linalg.qr(A)
print(f"Q orthogonal? {np.allclose(Q.T @ Q, np.eye(3))}")
print(f"Q @ R == A? {np.allclose(Q @ R, A)}")

In [ ]:
# Ex 6: PCA from scratch
np.random.seed(42)
X = np.random.multivariate_normal([2,3], [[2,1.5],[1.5,3]], 300)
X_c = X - X.mean(axis=0)
C = np.cov(X_c.T)
vals, vecs = np.linalg.eigh(C)
idx = np.argsort(vals)[::-1]
vals, vecs = vals[idx], vecs[:, idx]
X_proj = X_c @ vecs[:, :1]

plt.figure(figsize=(8, 6))
plt.scatter(X_c[:,0], X_c[:,1], alpha=0.3, s=10)
for i in range(2):
    v = vecs[:,i] * vals[i] * 0.5
    plt.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, linewidth=3, label=f'PC{i+1}')
plt.legend(); plt.axis('equal'); plt.grid(True); plt.title('PCA'); plt.show()

In [ ]:
# Ex 7: Low-rank approximation
np.random.seed(42)
A = np.random.randn(10, 10)
U, s, Vt = np.linalg.svd(A)
for k in [1, 3, 5]:
    A_k = U[:,:k] @ np.diag(s[:k]) @ Vt[:k,:]
    err = np.linalg.norm(A - A_k, 'fro')
    print(f"Rank-{k}: Frobenius error = {err:.4f}")

In [ ]:
# Ex 8: Gram-Schmidt
def gram_schmidt(V):
    n = V.shape[1]
    Q = np.zeros_like(V, dtype=float)
    for i in range(n):
        q = V[:, i].astype(float)
        for j in range(i):
            q -= np.dot(Q[:, j], V[:, i]) * Q[:, j]
        Q[:, i] = q / np.linalg.norm(q)
    return Q

V = np.array([[1,1,0],[1,0,1],[0,1,1]])
Q = gram_schmidt(V)
print(f"Q^T @ Q = I? {np.allclose(Q.T @ Q, np.eye(3))}")

In [ ]:
# Ex 9: Matrix power
A = np.array([[2,1],[0,3]])
A10_direct = np.linalg.matrix_power(A, 10)
vals, Q = np.linalg.eig(A)
A10_eigen = Q @ np.diag(vals**10) @ np.linalg.inv(Q)
print(f"Direct:\n{A10_direct}")
print(f"Eigen:\n{np.round(np.real(A10_eigen))}")
print(f"Match? {np.allclose(A10_direct, np.real(A10_eigen))}")

In [ ]:
# Ex 10: Cholesky
np.random.seed(42)
A = np.random.randn(3, 3)
S = A.T @ A  # Guaranteed PD
L = np.linalg.cholesky(S)
b = np.array([1.0, 2.0, 3.0])
# Solve Sx = b via L L^T x = b
y = linalg.solve_triangular(L, b, lower=True)
x = linalg.solve_triangular(L.T, y, lower=False)
print(f"x = {np.round(x, 4)}")
print(f"Verify: S @ x ≈ b? {np.allclose(S @ x, b)}")

In [ ]:
# Ex 11: Spectral normalization
np.random.seed(42)
for i in range(5):
    W = np.random.randn(100, 100)
    sn = np.linalg.norm(W, 2)
    W_norm = W / sn
    print(f"W{i+1}: spectral norm = {sn:.4f}, after normalization: {np.linalg.norm(W_norm, 2):.4f}")

In [ ]:
# Ex 12: Condition number
well = np.eye(3) + 0.1 * np.random.randn(3, 3)
ill = np.array([[1, 1], [1, 1.0001]])
print(f"Well-conditioned κ: {np.linalg.cond(well):.2f}")
print(f"Ill-conditioned κ: {np.linalg.cond(ill):.2f}")

b = np.array([2.0, 2.0])
b_noisy = b + 1e-4 * np.random.randn(2)
x1 = np.linalg.solve(ill, b)
x2 = np.linalg.solve(ill, b_noisy)
print(f"Tiny noise in b → large change in x: {np.linalg.norm(x1 - x2):.4f}")

In [ ]:
# Ex 13: Simple recommendation system
R = np.array([[5,3,0,1],[4,0,0,1],[1,1,0,5],[1,0,0,4],[0,1,5,4]], dtype=float)
# Fill missing (0) with row mean
R_filled = R.copy()
for i in range(R.shape[0]):
    mask = R[i] > 0
    R_filled[i, ~mask] = R[i, mask].mean()

U, s, Vt = np.linalg.svd(R_filled)
k = 2
R_pred = U[:,:k] @ np.diag(s[:k]) @ Vt[:k,:]
print(f"Predicted ratings:\n{np.round(R_pred, 1)}")

In [ ]:
# Ex 14: Variance explained
np.random.seed(42)
X = np.random.randn(200, 5) @ np.diag([5, 3, 1, 0.5, 0.1])
X_c = X - X.mean(axis=0)
vals = np.linalg.eigvalsh(np.cov(X_c.T))[::-1]
cumvar = np.cumsum(vals) / vals.sum()
n95 = np.argmax(cumvar >= 0.95) + 1
print(f"Components for 95% variance: {n95}")
plt.plot(range(1, 6), cumvar, 'o-'); plt.axhline(0.95, color='r', ls='--')
plt.xlabel('Components'); plt.ylabel('Cum. Variance'); plt.title('Scree Plot'); plt.show()

In [ ]:
# Ex 15: Trace/Det = sum/product of eigenvalues
for i in range(5):
    A = np.random.randn(4, 4)
    vals = np.linalg.eigvals(A)
    print(f"Matrix {i+1}: tr={np.trace(A):.4f} vs Σλ={np.sum(vals).real:.4f} | "
          f"det={np.linalg.det(A):.4f} vs Πλ={np.prod(vals).real:.4f}")